# ĐỒ ÁN MÔN BIG DATA: PHÂN TÍCH HÀNH VI KHÁCH HÀNG E-COMMERCE
## NOTEBOOK 01: KHÁM PHÁ DỮ LIỆU (EDA) & PHÂN TÍCH PHỄU CHUYỂN ĐỔI (FUNNEL ANALYSIS)
---
- **Học phần:** Dữ liệu Lớn (Big Data Analytics)
- **Bộ dữ liệu:** `bigquery-public-data.ga4_obfuscated_sample_ecommerce`
- **Công nghệ:** Google Cloud BigQuery SQL, Python (Pandas, Plotly, Recharts)
- **Nội dung:** Khám phá lưu lượng truy cập, bóc tách dữ liệu bán cấu trúc (UNNEST), và xây dựng phễu mua hàng 4 bước tuần tự theo User.

### 1. Cài đặt Môi trường & Khởi tạo BigQuery Service

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Thêm đường dẫn thư mục gốc vào sys.path
ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.bq_client import BigQueryService

# Khởi tạo dịch vụ BigQuery
bq = BigQueryService()
print("✓ Khởi tạo BigQuery Service thành công!")

### 2. Thực thi SQL 01: Khám phá Tổng quan (EDA) & Hiệu suất Kênh Tiếp thị

In [2]:
# Đọc và thực thi câu lệnh SQL 01_eda_overview.sql
query_eda = bq.read_sql_file("01_eda_overview.sql")
df_eda = bq.query_to_dataframe(query_eda, cache_name="eda_overview")

print("=== BẢNG THỐNG KÊ KÊNH TIẾP THỊ (TRAFFIC MEDIUMS) ===")
display(df_eda)

#### Trực quan hóa Doanh thu và Lượng người dùng theo Kênh tiếp thị

In [3]:
fig_eda = px.bar(
    df_eda,
    x="traffic_medium",
    y="total_revenue",
    color="conversion_rate",
    color_continuous_scale="Viridis",
    labels={"total_revenue": "Tổng Doanh thu ($ USD)", "traffic_medium": "Kênh Tiếp thị", "conversion_rate": "CR (%)"},
    title="Doanh thu và Tỷ lệ Chuyển đổi theo Kênh Tiếp thị (GA4 BigQuery)"
)
fig_eda.update_layout(template="plotly_dark")
fig_eda.show()

### 3. Thực thi SQL 02: Phân tích Phễu Mua hàng 4 bước (Funnel Analysis)
**Nguyên tắc phễu:**
$$\text{View Item} \longrightarrow \text{Add to Cart} \longrightarrow \text{Begin Checkout} \longrightarrow \text{Purchase}$$
- Đếm trên cùng 1 User (`COUNT(DISTINCT user_pseudo_id)`)
- Bước sau luôn $\le$ bước trước.

In [4]:
query_funnel = bq.read_sql_file("02_funnel_analysis.sql")
df_funnel = bq.query_to_dataframe(query_funnel, cache_name="funnel_analysis")

print("=== BẢNG THỐNG KÊ PHỄU THEO THIẾT BỊ (DESKTOP VS MOBILE VS TABLET) ===")
display(df_funnel)

#### Trực quan hóa Biểu đồ Phễu Chuyển đổi (Funnel Chart)

In [5]:
stages = ["1. View Item", "2. Add to Cart", "3. Begin Checkout", "4. Purchase"]
values = [
    df_funnel["step_1_view_item_users"].sum(),
    df_funnel["step_2_add_to_cart_users"].sum(),
    df_funnel["step_3_begin_checkout_users"].sum(),
    df_funnel["step_4_purchase_users"].sum()
]

fig_funnel = go.Figure(go.Funnel(
    y=stages,
    x=values,
    textinfo="value+percent initial+percent previous",
    marker=dict(color=["#3B82F6", "#0EA5E9", "#F59E0B", "#10B981"])
))
fig_funnel.update_layout(title="Phễu Chuyển đổi Tổng thể (Toàn bộ Thiết bị)", template="plotly_dark")
fig_funnel.show()